# ML-Filtered SMA Trading

This notebook combines the ML training pattern with the backtesting-framework pattern. It trains a CUDA cuML classifier on pre-2020 Quant Warehouse optimal-trade `side` labels, generates fixed post-2020 daily ML predictions, injects those predictions directly into `backtesting.py`, and then optimizes SMA crossover lookbacks while keeping the ML predictions unchanged.

The data, target labels, technical indicators, and ML prediction columns stay in memory. No temporary CSV or parquet handoff is used.

In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import itertools
import random
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "quant_orchestrator").exists():
            return candidate
    raise RuntimeError(f"Could not find quant-orchestrator repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
for path in (REPO_ROOT, REPO_ROOT.parent / "quant-warehouse"):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

warnings.filterwarnings("ignore", category=UserWarning)

from quant_warehouse import Warehouse
from quant_warehouse.feature_engineering import compute_features_worldclass
from quant_warehouse.target_engineering import build_label_panel

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

from backtesting import Backtest, Strategy
from quant_orchestrator.platforms.backtesting_frameworks.reporting import build_common_summary, normalize_equity_curve
from quant_orchestrator.platforms.backtesting_frameworks.shared import combine_equity_curves, equal_notional_capital

Loading BokehJS ...

In [2]:
SYMBOLS = ["AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "TSLA"]
PROVIDER = "yfinance"
TRAIN_START = "1990-01-01"
TRAIN_END = "2019-12-31"
BACKTEST_START = "2020-01-01"
BACKTEST_END = None
LABEL_K_PARAMS = {"YE": list(range(1, 13))}
ML_FEATURE_COLUMNS = ["open", "high", "low", "close", "volume"]
OHLCV_COLUMNS = ["open", "high", "low", "close", "volume"]
FAST_WINDOWS = [5, 10, 20, 50]
SLOW_WINDOWS = [60, 100, 150, 200]
CAPITAL_BASE = 100_000.0
RANDOM_SEED = 20260629

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

config = pd.DataFrame(
    [
        {"setting": "symbols", "value": ", ".join(SYMBOLS)},
        {"setting": "provider", "value": PROVIDER},
        {"setting": "ml_train_window", "value": f"{TRAIN_START} to {TRAIN_END}"},
        {"setting": "backtest_window", "value": f"{BACKTEST_START} to latest warehouse row"},
        {"setting": "target", "value": f"optimal side labels {LABEL_K_PARAMS}"},
        {"setting": "sma_fast_grid", "value": FAST_WINDOWS},
        {"setting": "sma_slow_grid", "value": SLOW_WINDOWS},
    ]
)
display(config)

,setting,value
0,symbols,"AAPL, MSFT, NVDA, AMZN, META, GOOGL, TSLA"
1,provider,yfinance
2,ml_train_window,1990-01-01 to 2019-12-31
3,backtest_window,2020-01-01 to latest warehouse row
4,target,"optimal side labels {'YE': [1, 2, 3, 4, 5, 6, ..."
5,sma_fast_grid,"[5, 10, 20, 50]"
6,sma_slow_grid,"[60, 100, 150, 200]"


## Load Warehouse Data, Features, And Labels

The model trains only on labels before 2020. The backtest starts in 2020. Quant Warehouse computes the SMA features; `backtesting.py` only consumes already prepared columns.

In [3]:
def normalize_price_frame(prices: pd.DataFrame) -> pd.DataFrame:
    frame = prices.rename(columns=str.lower).copy()
    frame.index = pd.to_datetime(frame.index).tz_localize(None)
    frame = frame.sort_index()
    missing = [column for column in OHLCV_COLUMNS if column not in frame.columns]
    if missing:
        raise ValueError(f"Missing OHLCV columns: {missing}")
    return frame.loc[:, OHLCV_COLUMNS].astype(float).dropna(subset=["open", "high", "low", "close"])


def load_feature_frame(symbol: str) -> pd.DataFrame:
    prices = Warehouse().read_prices(symbol, provider=PROVIDER, start=TRAIN_START, end=BACKTEST_END)
    frame = normalize_price_frame(prices)
    features = compute_features_worldclass(frame.copy())
    features.index = pd.to_datetime(features.index).tz_localize(None)
    return features.sort_index()


def build_symbol_labels(symbol: str, feature_frame: pd.DataFrame) -> pd.DataFrame:
    train_prices = feature_frame.loc[:TRAIN_END, OHLCV_COLUMNS].copy()
    labels = build_label_panel(
        {symbol: train_prices},
        k_params=LABEL_K_PARAMS,
        solver_mode="period_top_k",
        add_rank_labels=True,
        deduplicate=True,
        max_workers=1,
    ).reset_index()
    labels["date"] = pd.to_datetime(labels["date"]).dt.tz_localize(None)
    labels["side"] = labels["side"].astype(str).str.lower()
    labels = labels[labels["side"].isin(["long", "short"])].copy()
    labels["side_code"] = labels["side"].map({"long": 0, "short": 1}).astype("int32")
    labels["symbol"] = symbol
    return labels


feature_frames = {symbol: load_feature_frame(symbol) for symbol in SYMBOLS}
label_frames = {symbol: build_symbol_labels(symbol, feature_frames[symbol]) for symbol in SYMBOLS}

label_summary = pd.DataFrame(
    [
        {
            "symbol": symbol,
            "feature_rows": len(feature_frames[symbol]),
            "label_rows_pre_2020": len(label_frames[symbol]),
            "long_rate": (label_frames[symbol]["side"] == "long").mean(),
            "first_feature_date": feature_frames[symbol].index.min().date(),
            "last_feature_date": feature_frames[symbol].index.max().date(),
        }
        for symbol in SYMBOLS
    ]
)
display(label_summary)

,symbol,feature_rows,label_rows_pre_2020,long_rate,first_feature_date,last_feature_date
0,AAPL,9188,864,0.532407,1990-01-02,2026-06-26
1,MSFT,9188,852,0.535211,1990-01-02,2026-06-26
2,NVDA,6899,602,0.551495,1999-01-22,2026-06-26
3,AMZN,7324,659,0.544765,1997-05-15,2026-06-26
4,META,3546,233,0.562232,2012-05-18,2026-06-26
5,GOOGL,5498,451,0.541020,2004-08-19,2026-06-26
6,TSLA,4023,292,0.513699,2010-06-29,2026-06-26


In [4]:
def build_training_table() -> pd.DataFrame:
    rows = []
    for symbol in SYMBOLS:
        labels = label_frames[symbol]
        features = feature_frames[symbol].loc[:, ML_FEATURE_COLUMNS].copy()
        joined = labels.merge(
            features.reset_index().rename(columns={"index": "date"}),
            on="date",
            how="inner",
        )
        joined["symbol"] = symbol
        rows.append(joined)
    table = pd.concat(rows, ignore_index=True).dropna(subset=ML_FEATURE_COLUMNS + ["side_code"])
    return table.sort_values(["date", "symbol"]).reset_index(drop=True)


training_table = build_training_table()
display(training_table[["date", "symbol", "side", "rank_y", *ML_FEATURE_COLUMNS]].head(10))
display(
    pd.DataFrame(
        [
            {
                "training_rows": len(training_table),
                "first_train_label": training_table["date"].min().date(),
                "last_train_label": training_table["date"].max().date(),
                "long_rate": (training_table["side"] == "long").mean(),
            }
        ]
    )
)

,date,symbol,side,rank_y,open,high,low,close,volume
0,1990-01-03,AAPL,short,0.298032,0.265710,0.265710,0.262213,0.262213,207995200.0
1,1990-01-15,MSFT,long,0.986502,0.362143,0.367423,0.360032,0.363727,62467200.0
2,1990-01-29,AAPL,long,0.818866,0.230748,0.234244,0.224629,0.232496,119929600.0
3,1990-01-29,AAPL,short,0.298032,0.230748,0.234244,0.224629,0.232496,119929600.0
4,1990-02-08,AAPL,long,0.737847,0.232496,0.234244,0.225503,0.230748,186636800.0
5,1990-03-26,AAPL,long,0.688657,0.298132,0.304271,0.294625,0.296379,128060800.0
6,1990-04-02,AAPL,long,0.185185,0.280596,0.284980,0.277088,0.282349,148769600.0
7,1990-04-16,AAPL,long,0.737847,0.305148,0.310408,0.303394,0.306901,226889600.0
8,1990-04-16,MSFT,long,0.922535,0.515236,0.521571,0.507845,0.513125,39470400.0
9,1990-04-17,AAPL,short,0.265625,0.303394,0.305148,0.299886,0.303394,131107200.0


,training_rows,first_train_label,last_train_label,long_rate
0,3953,1990-01-03,2019-12-31,0.539337


## Train Pre-2020 ML Model And Generate Fixed 2020+ Predictions

The ML model is trained once. The optimization loop below does not retrain the model and does not change predictions.

In [5]:
def train_cuml_side_classifier(training_table: pd.DataFrame):
    import cudf
    from cuml.ensemble import RandomForestClassifier

    scaler = StandardScaler()
    x_train_np = scaler.fit_transform(training_table[ML_FEATURE_COLUMNS]).astype("float32")
    y_train_np = training_table["side_code"].astype("int32").to_numpy()

    model = RandomForestClassifier(
        n_estimators=256,
        max_depth=10,
        random_state=RANDOM_SEED,
        n_streams=1,
    )
    started = perf_counter()
    model.fit(
        cudf.DataFrame(pd.DataFrame(x_train_np, columns=ML_FEATURE_COLUMNS)),
        cudf.Series(y_train_np),
    )
    train_seconds = perf_counter() - started
    pred = model.predict(cudf.DataFrame(pd.DataFrame(x_train_np, columns=ML_FEATURE_COLUMNS))).to_numpy().astype(int)
    proba = model.predict_proba(cudf.DataFrame(pd.DataFrame(x_train_np, columns=ML_FEATURE_COLUMNS))).to_numpy()[:, 1]
    metrics = {
        "train_accuracy": accuracy_score(y_train_np, pred),
        "train_macro_f1": f1_score(y_train_np, pred, average="macro", zero_division=0),
        "train_short_roc_auc": roc_auc_score(y_train_np, proba),
        "train_seconds": train_seconds,
    }
    return model, scaler, metrics


ml_model, ml_scaler, train_metrics = train_cuml_side_classifier(training_table)
display(pd.DataFrame([train_metrics]))

[08:50:17] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


,train_accuracy,train_macro_f1,train_short_roc_auc,train_seconds
0,0.690362,0.658758,0.792118,1.027974


In [6]:
def predict_symbol_frame(symbol: str) -> pd.DataFrame:
    import cudf

    frame = feature_frames[symbol].loc[BACKTEST_START:, :].copy()
    frame = frame.dropna(subset=ML_FEATURE_COLUMNS).copy()
    x_np = ml_scaler.transform(frame[ML_FEATURE_COLUMNS]).astype("float32")
    x_gpu = cudf.DataFrame(pd.DataFrame(x_np, columns=ML_FEATURE_COLUMNS))
    pred_code = ml_model.predict(x_gpu).to_numpy().astype(int)
    proba_short = ml_model.predict_proba(x_gpu).to_numpy()[:, 1]
    frame["ml_pred_side"] = np.where(pred_code == 0, "long", "short")
    frame["ml_long_filter"] = (pred_code == 0).astype(int)
    frame["ml_short_probability"] = proba_short
    frame["ml_long_probability"] = 1.0 - proba_short
    return frame


prediction_frames = {symbol: predict_symbol_frame(symbol) for symbol in SYMBOLS}
prediction_summary = pd.DataFrame(
    [
        {
            "symbol": symbol,
            "prediction_rows": len(prediction_frames[symbol]),
            "first_prediction_date": prediction_frames[symbol].index.min().date(),
            "last_prediction_date": prediction_frames[symbol].index.max().date(),
            "ml_long_rate": prediction_frames[symbol]["ml_long_filter"].mean(),
        }
        for symbol in SYMBOLS
    ]
)
display(prediction_summary)
display(prediction_frames["AAPL"][["open", "close", "ml_pred_side", "ml_long_probability", "ml_short_probability"]].head(10))

,symbol,prediction_rows,first_prediction_date,last_prediction_date,ml_long_rate
0,AAPL,1629,2020-01-02,2026-06-26,0.510743
1,MSFT,1629,2020-01-02,2026-06-26,0.767342
2,NVDA,1629,2020-01-02,2026-06-26,0.252302
3,AMZN,1629,2020-01-02,2026-06-26,0.559239
4,META,1629,2020-01-02,2026-06-26,0.877225
5,GOOGL,1629,2020-01-02,2026-06-26,0.850829
6,TSLA,1629,2020-01-02,2026-06-26,0.181093


,open,close,ml_pred_side,ml_long_probability,ml_short_probability
date,,,,,
2020-01-02,71.344047,72.333870,long,0.776043,0.223957
2020-01-03,71.563190,71.630623,long,0.635141,0.364859
2020-01-06,70.754014,72.201408,long,0.734879,0.265121
2020-01-07,72.211033,71.861832,long,0.700826,0.299174
2020-01-08,71.565629,73.017845,long,0.776043,0.223957
2020-01-09,73.993195,74.568787,long,0.629933,0.370067
2020-01-10,74.802380,74.737350,long,0.753124,0.246876
2020-01-13,75.052863,76.334084,long,0.770785,0.229215
2020-01-14,76.271471,75.303322,long,0.629933,0.370067


## Backtest SMA Crossover With Fixed ML Filter

For each fast/slow SMA pair, the strategy enters only when both conditions are true:

- `fast_sma > slow_sma`
- the fixed ML prediction says the side is `long`

The SMA pair changes during optimization. The ML predictions do not.

In [7]:
def build_backtesting_py_frame(frame: pd.DataFrame, *, fast_window: int, slow_window: int) -> pd.DataFrame:
    fast_col = f"SMA{fast_window}"
    slow_col = f"SMA{slow_window}"
    missing = [column for column in (fast_col, slow_col) if column not in frame.columns]
    if missing:
        raise ValueError(f"Quant Warehouse feature frame is missing columns: {missing}")
    out = frame.loc[:, OHLCV_COLUMNS + [fast_col, slow_col, "ml_long_filter", "ml_long_probability"]].copy()
    out = out.dropna(subset=[fast_col, slow_col, "ml_long_filter"]).copy()
    out["FastAboveSlow"] = (out[fast_col] > out[slow_col]).astype(int)
    out["MLFilter"] = out["ml_long_filter"].astype(int)
    out["Signal"] = ((out["FastAboveSlow"] == 1) & (out["MLFilter"] == 1)).astype(int)
    out = out.rename(
        columns={
            "open": "Open",
            "high": "High",
            "low": "Low",
            "close": "Close",
            "volume": "Volume",
        }
    )
    return out


class MlFilteredSmaStrategy(Strategy):
    trade_fraction = 0.95

    def init(self):
        pass

    def next(self):
        bullish_sma = bool(self.data.FastAboveSlow[-1])
        ml_allows_long = bool(self.data.MLFilter[-1])
        if bullish_sma and ml_allows_long and not self.position:
            self.buy(size=self.trade_fraction)
        elif self.position and (not bullish_sma or not ml_allows_long):
            self.position.close()


def run_symbol_backtest(symbol: str, *, fast_window: int, slow_window: int, capital: float, use_ml_filter: bool = True):
    frame = build_backtesting_py_frame(prediction_frames[symbol], fast_window=fast_window, slow_window=slow_window)
    if not use_ml_filter:
        frame["MLFilter"] = 1
        frame["Signal"] = frame["FastAboveSlow"].astype(int)
    started = perf_counter()
    stats = Backtest(
        frame,
        MlFilteredSmaStrategy,
        cash=capital,
        commission=0.0,
        trade_on_close=False,
        exclusive_orders=True,
    ).run()
    elapsed = perf_counter() - started
    equity = stats["_equity_curve"]["Equity"].rename("portfolio_value")
    trades = stats.get("# Trades", 0)
    summary = build_common_summary(
        framework="backtesting.py",
        symbol=symbol,
        equity=equity,
        elapsed_seconds=elapsed,
        bars=len(frame),
        trades=int(trades) if pd.notna(trades) else 0,
    )
    summary["fast_window"] = fast_window
    summary["slow_window"] = slow_window
    summary["use_ml_filter"] = bool(use_ml_filter)
    summary["ml_long_rate"] = float(frame["MLFilter"].mean())
    summary["sma_long_rate"] = float(frame["FastAboveSlow"].mean())
    summary["combined_signal_rate"] = float(frame["Signal"].mean())
    return stats, summary, normalize_equity_curve(equity)


def run_portfolio_backtest(*, fast_window: int, slow_window: int, use_ml_filter: bool = True) -> dict[str, object]:
    symbol_capital = equal_notional_capital(CAPITAL_BASE, len(SYMBOLS))
    symbol_rows = []
    curves = []
    for symbol in SYMBOLS:
        _, summary, equity = run_symbol_backtest(
            symbol,
            fast_window=fast_window,
            slow_window=slow_window,
            capital=symbol_capital,
            use_ml_filter=use_ml_filter,
        )
        row = summary.iloc[0].to_dict()
        row["symbol"] = symbol
        symbol_rows.append(row)
        curves.append(equity)
    portfolio_equity = combine_equity_curves(curves)
    portfolio_summary = build_common_summary(
        framework="backtesting.py",
        symbol="MAG7",
        equity=portfolio_equity,
        elapsed_seconds=sum(row["elapsed_seconds"] for row in symbol_rows),
        bars=sum(row["bars"] for row in symbol_rows),
        trades=sum(row["trades"] for row in symbol_rows),
    )
    portfolio_summary["fast_window"] = fast_window
    portfolio_summary["slow_window"] = slow_window
    portfolio_summary["use_ml_filter"] = bool(use_ml_filter)
    portfolio_summary["performance_score"] = portfolio_summary["total_return"] + portfolio_summary["max_drawdown"]
    return {
        "portfolio_summary": portfolio_summary,
        "symbol_summary": pd.DataFrame(symbol_rows),
        "portfolio_equity": portfolio_equity,
    }

In [8]:
parameter_grid = [(fast, slow) for fast, slow in itertools.product(FAST_WINDOWS, SLOW_WINDOWS) if fast < slow]
optimization_rows = []
optimization_runs = {}

for fast_window, slow_window in parameter_grid:
    run = run_portfolio_backtest(fast_window=fast_window, slow_window=slow_window, use_ml_filter=True)
    row = run["portfolio_summary"].iloc[0].to_dict()
    optimization_rows.append(row)
    optimization_runs[(fast_window, slow_window)] = run

optimization_table = pd.DataFrame(optimization_rows).sort_values(
    ["performance_score", "sharpe", "total_return"],
    ascending=[False, False, False],
).reset_index(drop=True)

display(optimization_table[["fast_window", "slow_window", "total_return", "max_drawdown", "sharpe", "trades", "performance_score", "elapsed_seconds"]])

,fast_window,slow_window,total_return,max_drawdown,sharpe,trades,performance_score,elapsed_seconds
0,20,200,1.2050,-0.2564,0.9967,976,0.9486,0.1045
1,50,200,1.2060,-0.2608,0.9691,958,0.9452,0.1055
2,20,150,1.0972,-0.2564,0.9677,904,0.8408,0.1040
3,10,200,1.0966,-0.2584,0.9582,973,0.8382,0.1055
4,5,200,1.0577,-0.2500,0.9442,981,0.8077,0.1048
5,5,60,1.0082,-0.2154,1.0117,894,0.7928,0.1062
6,20,100,1.0405,-0.2570,0.9439,875,0.7835,0.1022
7,10,100,1.0174,-0.2405,0.9453,880,0.7769,0.1029
8,50,150,1.0114,-0.2607,0.8829,901,0.7507,0.1039
9,10,150,1.0016,-0.2561,0.9170,912,0.7455,0.1050


In [9]:
best = optimization_table.iloc[0]
best_fast = int(best["fast_window"])
best_slow = int(best["slow_window"])
ml_best_run = optimization_runs[(best_fast, best_slow)]
baseline_run = run_portfolio_backtest(fast_window=best_fast, slow_window=best_slow, use_ml_filter=False)

comparison = pd.concat(
    [
        ml_best_run["portfolio_summary"].assign(strategy="SMA + fixed ML long filter"),
        baseline_run["portfolio_summary"].assign(strategy="SMA only baseline"),
    ],
    ignore_index=True,
)
comparison = comparison[
    [
        "strategy",
        "fast_window",
        "slow_window",
        "total_return",
        "annualized_return",
        "max_drawdown",
        "sharpe",
        "trades",
        "bars",
        "elapsed_seconds",
    ]
]
display(Markdown(f"## Best SMA Parameters With Fixed ML Predictions: {best_fast}/{best_slow}"))
display(comparison)

display(Markdown("### Symbol-Level Results For Best ML-Filtered Run"))
display(
    ml_best_run["symbol_summary"][[
        "symbol",
        "total_return",
        "max_drawdown",
        "sharpe",
        "trades",
        "ml_long_rate",
        "sma_long_rate",
        "combined_signal_rate",
    ]].sort_values("total_return", ascending=False)
)

## Best SMA Parameters With Fixed ML Predictions: 20/200

,strategy,fast_window,slow_window,total_return,annualized_return,max_drawdown,sharpe,trades,bars,elapsed_seconds
0,SMA + fixed ML long filter,20,200,1.2050,0.1301,-0.2564,0.9967,976,11403,0.1045
1,SMA only baseline,20,200,3.7615,0.2730,-0.3657,0.9700,44,11403,0.0982


### Symbol-Level Results For Best ML-Filtered Run

,symbol,total_return,max_drawdown,sharpe,trades,ml_long_rate,sma_long_rate,combined_signal_rate
5,GOOGL,2.0576,-0.3425,0.8843,120,0.850829,0.767956,0.653161
4,META,1.9109,-0.3665,0.7531,111,0.877225,0.687538,0.610190
2,NVDA,1.3964,-0.5058,0.6743,93,0.252302,0.834868,0.217925
0,AAPL,1.0368,-0.2592,0.7074,174,0.510743,0.796808,0.443831
3,AMZN,0.9418,-0.2216,0.6382,182,0.559239,0.717618,0.373235
6,TSLA,0.6035,-0.4985,0.4133,95,0.181093,0.664825,0.144874
1,MSFT,0.4880,-0.2221,0.4458,201,0.767342,0.733579,0.559239


In [10]:
readout_lines = [
    "## Notebook Readout",
    f"- Trained one CUDA cuML side classifier on {len(training_table):,} pre-2020 optimal-label rows across MAG7.",
    f"- Generated fixed daily ML predictions for {sum(len(frame) for frame in prediction_frames.values()):,} post-2020 symbol-days.",
    f"- Optimized {len(parameter_grid)} SMA fast/slow combinations while keeping ML predictions fixed.",
    f"- Best fixed-filter parameters were {best_fast}/{best_slow} by total_return + max_drawdown.",
]
ml_row = comparison[comparison["strategy"].eq("SMA + fixed ML long filter")].iloc[0]
base_row = comparison[comparison["strategy"].eq("SMA only baseline")].iloc[0]
readout_lines.append(
    f"- ML-filtered portfolio total return was {ml_row['total_return']:.2%} with max drawdown {ml_row['max_drawdown']:.2%}."
)
readout_lines.append(
    f"- SMA-only baseline at the same windows returned {base_row['total_return']:.2%} with max drawdown {base_row['max_drawdown']:.2%}."
)
readout_lines.append(
    "- This notebook intentionally keeps the model fixed during strategy optimization to avoid tuning the classifier and SMA parameters together."
)
display(Markdown("\n".join(readout_lines)))

## Notebook Readout
- Trained one CUDA cuML side classifier on 3,953 pre-2020 optimal-label rows across MAG7.
- Generated fixed daily ML predictions for 11,403 post-2020 symbol-days.
- Optimized 16 SMA fast/slow combinations while keeping ML predictions fixed.
- Best fixed-filter parameters were 20/200 by total_return + max_drawdown.
- ML-filtered portfolio total return was 120.50% with max drawdown -25.64%.
- SMA-only baseline at the same windows returned 376.15% with max drawdown -36.57%.
- This notebook intentionally keeps the model fixed during strategy optimization to avoid tuning the classifier and SMA parameters together.